<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🛡️ Lab 06: Red-Team Your Hosted MAF Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Confirm that security testing targets the API, not the framework — same attacks, same methodology
  </p>
</div>

**What We Learn:** Red teaming works the same way regardless of agent implementation — the attack probes the external Responses API, not the internal code. Security posture depends on the model and instructions, not the framework.

---


## 🧳 The Contoso Travel Story

The Contoso Travel hosted MAF agent has passed quality and safety evaluations. But the security team has a final gate: adversarial testing. They want to know if the MAF implementation — with its custom Python code and tool functions — introduces new attack surfaces that didn't exist in the prompted agent.

- **The Problem:** Does the MAF implementation change the agent's vulnerability profile? Could custom code introduce new attack surfaces — perhaps a tool function that leaks internal data, or business logic that can be manipulated through crafted inputs?
- **This Lab Solves:** Running the same red team scan against the hosted agent to verify that security posture is consistent and framework-independent. The red teaming service sends adversarial prompts through the Responses API — it doesn't know or care how the agent processes them internally.

## 1. Setup

---


In [ ]:
# Setup: load env vars and connect to Foundry with red team imports
import os
import time
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    RedTeam,
    AzureOpenAIModelConfiguration,
    AttackStrategy,
    RiskCategory,
)

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")
model_endpoint = os.environ["MODEL_ENDPOINT"]
model_api_key = os.environ["MODEL_API_KEY"]

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print(f"✅ Connected to Microsoft Foundry")
print(f"   Model: {model_name}")
print(f"   Model Endpoint: {model_endpoint[:30]}...")

## 2. Why Red Teaming Is Agent-Agnostic

Think of red teaming as hiring a professional burglar to test your locks. The burglar tries the front door (the Responses API) — they don't care whether the lock mechanism inside is a deadbolt (prompted agent), a smart lock (MAF agent), or a combination lock (LangGraph agent). The test is the same.

```
┌──────────────────┐     ┌──────────────┐     ┌───────────────────┐
│  Red Team        │     │ Responses API│     │  Any Agent        │
│  (adversarial    │ ──▶ │ (the "front  │ ──▶ │  • Prompted       │
│   prompts)       │     │  door")      │     │  • MAF BaseAgent  │
│                  │     │              │     │  • LangGraph      │
└──────────────────┘     └──────────────┘     └───────────────────┘
```

Red teaming sends adversarial prompts through the **Responses API** — the same external interface regardless of agent implementation. The red teaming service:

1. Generates adversarial inputs using attack strategies (jailbreak, crescendo, etc.)
2. Sends them to the model endpoint
3. Evaluates whether the responses violate safety boundaries

It has **zero visibility** into the agent's internal architecture. Whether your agent uses `BaseAgent` with custom Python code or a simple `PromptAgentDefinition`, the attack surface is the same: the model endpoint and its system instructions.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #e91e63; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>🔐 Security Insight:</b> If your prompted agent and hosted MAF agent use the same model and similar instructions, their red team results should be comparable. Any differences reveal how custom code or tool functions affect the security posture — valuable signal for hardening your agent.
</div>

---


## 3. Configure Red Team Scan

We configure the same attack strategies and risk categories used in the prompted agents lab.

---


In [ ]:
# Configure red team target model and attack parameters
target_config = AzureOpenAIModelConfiguration(
    model_deployment_name=model_name
)

red_team = RedTeam(
    target=target_config,
    risk_categories=[
        RiskCategory.VIOLENCE,
        RiskCategory.HATE_UNFAIRNESS,
    ],
    attack_strategies=[
        AttackStrategy.JAILBREAK,
        AttackStrategy.CRESCENDO,
    ],
    display_name="contoso-travel-maf-redteam",
    num_objectives=5,
)

print("✅ Red Team configuration created")
print(f"   Risk categories: Violence, Hate/Unfairness")
print(f"   Attack strategies: Jailbreak, Crescendo")
print(f"   Objectives: 5")
print(f"   Target model: {model_name}")

## 4. Run the Red Team Scan

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #9c27b0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>⏱️ Note:</b> Red team scans can take several minutes to complete depending on the number of risk categories and attack strategies.
</div>

---


In [ ]:
# Submit the red team scan to Foundry
print("🚀 Starting Red Team scan...")

red_team_response = project_client.beta.red_teams.create(
    red_team=red_team,
    headers={
        "model-endpoint": model_endpoint,
        "model-api-key": model_api_key,
    },
)

print(f"✅ Red Team scan created!")
print(f"   Scan name: {red_team_response.name}")
print(f"   Status: {red_team_response.status}")

In [ ]:
# Monitor scan progress
scan_name = red_team_response.name

while True:
    scan_status = project_client.beta.red_teams.get(name=scan_name)
    print(f"   ⏳ Status: {scan_status.status}")

    if scan_status.status in ["Completed", "Failed", "completed", "failed"]:
        break
    time.sleep(15)

print(f"\n{'✅' if 'omplete' in str(scan_status.status) else '❌'} Scan {scan_status.status}!")

## 5. Review Results

Let's examine what the red team scan found.

---


In [ ]:
# Display red team scan results summary
print(f"📊 Red Team Scan Results")
print(f"{'='*60}")
print(f"   Scan Name: {scan_name}")
print(f"   Status: {scan_status.status}")

print(f"\n📋 All Red Team Scans:")
for scan in project_client.beta.red_teams.list():
    print(f"   • {scan.name} — Status: {scan.status}")

## 6. Compare with Prompted Agent Results

If you completed Lab 06 in the prompted agents track, compare your findings:

| Dimension | Prompted Agent | Hosted MAF Agent |
|---|---|---|
| Attack strategies tested | Base64, Jailbreak | Jailbreak, Crescendo |
| Risk categories | Violence, Hate/Unfairness | Violence, Hate/Unfairness |
| Vulnerabilities found | *(your count)* | *(your count)* |
| Overall posture | *(your assessment)* | *(your assessment)* |

Key questions to consider:
- Are the vulnerability counts similar? If so, this confirms security depends on the **model and instructions**, not the framework.
- Did the crescendo attack strategy reveal anything different from the base64 strategy used in the prompted agents lab?
- Would custom tool functions introduce new attack surfaces not covered by model-level red teaming?

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #ff9800; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>⚠️ Important:</b> Model-level red teaming tests the model's response behavior. If your hosted agent has custom tool functions that access databases or external APIs, you should also perform application-level security testing (input validation, authorization, etc.) beyond what this red team scan covers.
</div>

---


## 7. View Results in the Foundry Portal

For detailed red team results:

1. Go to the [Microsoft Foundry portal](https://ai.azure.com)
2. Open your project
3. Navigate to **Evaluations** → **Red Teaming** results
4. Review individual attack attempts, model responses, and risk scores

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Tip:</b> The portal provides conversation-level detail showing exactly how the adversarial agent probed your model and how it responded. Compare the MAF scan results side-by-side with the prompted agent scan.
</div>


<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #4caf50; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>🔄 Local &amp; Deployed:</b> Red team scans work identically whether your agent runs locally or is deployed to Foundry Agent Service. The adversarial prompts target the model endpoint — they never interact with the agent's internal code or framework.
</div>

---

## 8. Cleanup

---


In [ ]:
# Close client connections and release resources
project_client.close()
credential.close()
print("✅ Clients closed")

## 9. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Understood why red teaming is agent-agnostic — attacks target the API, not the framework</li>
  <li>Configured and ran a red team scan with <b>Jailbreak</b> and <b>Crescendo</b> attack strategies</li>
  <li>Tested against <b>Violence</b> and <b>Hate/Unfairness</b> risk categories</li>
  <li>Compared results with the prompted agent scan to validate framework independence</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <b>💡 Key Takeaway:</b> Red teaming — like evaluation — is an API-level concern. The same attacks, methodologies, and risk categories apply regardless of whether your agent is prompted, hosted with MAF, or built with LangGraph. Security posture depends on the model, instructions, and content filters — not the framework.
</div>

🎉 **Congratulations!** You've completed the Hosted Agents (MAF) track:
1. ✅ **Tracing** — Auto-instrumented platform spans + custom business logic spans
2. ✅ **Evaluation** — Agent-agnostic quality and safety assessment
3. ✅ **Red Teaming** — Framework-independent adversarial security testing

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <b>➡️ Next:</b> Continue to the <b>Hosted Agents — LangGraph</b> track to see how the same observability patterns apply to graph-based agent architectures!
</div>

---
